# DriftWise: Adaptive Random Forest (ARF) baseline
Runs ARF on the same synthetic pattern-drift streams (seeds 10 to 19) and compares it with the results already saved for `ega_v2` and the retraining schedules.

**How to run:**
1. Run cell 1 (install). Then **Runtime > Restart session**.
2. Do NOT run cell 1 again. Run cell 2 (allow Google Drive access when asked).
3. ARF is slow in Python: expect about 15 to 25 minutes. Keep the tab open. It prints one line per seed and saves partial results as it goes.
4. Paste the printed tables back into the chat.

Needs `ega_pipeline.py` in `MyDrive/DriftWise/src/` and `ega_v2_vs_schedules_pattern_lag0.csv` in `MyDrive/DriftWise/results/`.

## 1. Install (run once, then Runtime > Restart session)

In [ ]:
!pip install river "numpy<2.3" -q

## 2. ARF baseline

In [ ]:
import sys, time, os
import numpy as np, pandas as pd
from scipy import stats
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/DriftWise/src")
import ega_pipeline as ep                       # for make_stream (the same streams as every other experiment)

try:
    from river import forest
    ARF = forest.ARFClassifier                  # Adaptive Random Forest (Gomes et al., 2017)
except Exception:
    from river import ensemble
    ARF = ensemble.AdaptiveRandomForestClassifier

def run_arf(X, y, oversample=1, start=4000, n_models=10, seed=42):
    """Prequential (test-then-train) ARF. It first learns the same first 4,000 records the other methods are
    trained on, then predicts each record BEFORE learning its label. oversample=3 shows every attack 3 times
    (a simple way to help ARF with the class imbalance)."""
    t0 = time.perf_counter()
    model = ARF(n_models=n_models, seed=seed)
    n = len(y)
    preds = np.zeros(n, dtype=int)
    for i in range(n):
        xi = {"f0": float(X[i, 0]), "f1": float(X[i, 1])}
        if i >= start:
            p = model.predict_one(xi)
            preds[i] = 0 if p is None else int(p)
        for _ in range(oversample if y[i] == 1 else 1):
            model.learn_one(xi, int(y[i]))
    ye, pe = y[5000:], preds[5000:]                 # same scoring window as every other method
    r0 = (pe[ye == 0] == 0).mean(); r1 = (pe[ye == 1] == 1).mean()
    return {"balanced_acc": (r0 + r1) / 2, "worst_class_recall": min(r0, r1), "seconds": time.perf_counter() - t0}

VARIANTS = {"ARF (default)": 1, "ARF (attacks shown 3x)": 3}
SEEDS = range(10, 20)                               # 10 fresh seeds; ARF is slow in pure Python
rows = []
os.makedirs("/content/drive/MyDrive/DriftWise/results", exist_ok=True)
for k, seed in enumerate(SEEDS, start=1):
    X, y = ep.make_stream(seed, "pattern_drift")
    for name, ov in VARIANTS.items():
        r = run_arf(X, y, oversample=ov)
        rows.append({"method": name, "seed": seed, **r})
    pd.DataFrame(rows).to_csv("/content/drive/MyDrive/DriftWise/results/arf_baseline_partial.csv", index=False)
    print(f"seed {k}/{len(SEEDS)} done | last run took {rows[-1]['seconds']:.0f}s")
arf = pd.DataFrame(rows)

# Results already computed earlier for the same seeds (pattern drift, labels arrive immediately)
ref = pd.read_csv("/content/drive/MyDrive/DriftWise/results/ega_v2_vs_schedules_pattern_lag0.csv")
ref = ref[ref.seed.isin(list(SEEDS))][["method", "seed", "balanced_acc", "worst_class_recall", "n_updates"]]
allr = pd.concat([arf[["method", "seed", "balanced_acc", "worst_class_recall", "seconds"]], ref], ignore_index=True)

pd.set_option("display.width", 250, "display.max_columns", 20, "display.max_colwidth", None)
cols = ["balanced_acc", "worst_class_recall", "n_updates", "seconds"]
print("\nMEAN OVER SEEDS", list(SEEDS), "(pattern drift, no label delay)")
display(allr.groupby("method")[[c for c in cols if c in allr.columns]].mean().round(3))

print("ega_v2 minus each ARF variant, paired by seed (mean [95% CI]):")
E = "ega_v2 (gated, keep improving)"
e = allr[allr.method == E].set_index("seed")
for name in VARIANTS:
    a = allr[allr.method == name].set_index("seed")
    for m in ["balanced_acc", "worst_class_recall"]:
        d = (e[m] - a[m]).dropna()
        sd = d.std(ddof=1)
        h = stats.t.ppf(0.975, len(d) - 1) * sd / np.sqrt(len(d)) if sd > 0 else 0.0
        print(f"  vs {name:26s} {m:19s} {d.mean():+.3f} [{d.mean()-h:+.3f}, {d.mean()+h:+.3f}]")

allr.to_csv("/content/drive/MyDrive/DriftWise/results/arf_baseline_comparison.csv", index=False)
print("\nSaved: arf_baseline_comparison.csv")